# Databricks / PySpark for Data Analysts/Engineers — Module 14: Working in the Databricks Workspace

For thirteen modules we wrote PySpark code that runs identically locally and in
Databricks. This module is different -- it doesn't teach new Spark syntax, just the
**Databricks platform itself**: what the workspace looks like, how to run notebooks on a
real cluster, and how to parameterize and schedule jobs. This is practical knowledge
needed to go from "I can write PySpark" to "I can work on a data team on Databricks".

## Table of Contents
1. [Anatomy of the Databricks Workspace](#sec1)
2. [Notebooks in Databricks](#sec2)
3. [Clusters and Compute](#sec3)
4. [`dbutils` -- the notebook's Swiss Army knife](#sec4)
5. [Widgets -- parameterizing notebooks](#sec5)
6. [Jobs and Workflows -- scheduling tasks](#sec6)
7. [A hands-on walkthrough of Databricks Free Edition](#sec7)
8. [Summary and exercises](#sec8)


> ⚠️ **This module is different from the previous ones**
>
> In Modules 1-13 we ran our code locally, because we were writing "plain" PySpark. This module describes parts of the Databricks platform (`dbutils`, widgets, the Jobs UI) that exist ONLY inside a real Databricks environment -- they cannot be run locally or tested in this notebook. That's why the code examples in this module are shown as snippets to read and try yourself in Databricks Free Edition (introduced in Module 1), rather than as runnable cells.

<a id="sec1"></a>
## 1. Anatomy of the Databricks Workspace

After logging into Databricks (Free Edition or a paid company workspace) you land in the
**Workspace** -- the main working environment. The sidebar contains a few key sections:

- **Workspace** -- a folder tree with your notebooks, SQL queries, and files; the
  equivalent of a file system, but for Databricks resources. You can create folders here,
  share notebooks with collaborators, and browse revision history.
- **Repos** (or **Git folders**) -- Git integration (GitHub, GitLab, Azure DevOps): you
  clone a repository straight into the workspace, so notebooks live in the same repo as
  the rest of the team's code, with full `git pull`/`push`/`commit` from the UI.
- **Catalog** (Catalog Explorer / Unity Catalog) -- a browser for every catalog, schema,
  and table, the concept we covered in Module 11. From here you can check a table's
  schema, its permissions, and its lineage without writing a single line of code.
- **Compute** -- the list of clusters and SQL Warehouses: this is where you create,
  start, stop, and monitor resource usage for them.
- **Workflows** (formerly Jobs) -- scheduling and orchestration: this is where you define
  WHEN and in WHAT ORDER your notebooks and scripts should run (section 6).

All these sections are connected: you run a notebook from Workspace on a cluster from
Compute, reading tables from Catalog, and you schedule the whole thing as a job in
Workflows.

> 🧱 **Free Edition vs. company workspace**
>
> Databricks Free Edition has a simplified navigation (fewer administrative settings), but all the sections above are present and work the same conceptually -- what you learn here carries straight over to a paid company workspace.

<a id="sec2"></a>
## 2. Notebooks in Databricks

Databricks notebooks resemble Jupyter (which we've used all course), but have a few
features Jupyter doesn't:

**Multi-language cells.** You set a notebook's default language once, but EVERY cell can
use a different language via *magic commands* at the top of the cell:

```
%python
# PySpark code (this course's default language)
df = spark.table("customers")

%sql
-- plain SQL, exactly like in the SQL/PostgreSQL course
SELECT segment, COUNT(*) FROM customers GROUP BY segment

%scala
// Scala code -- same SparkSession, different language
val df = spark.table("customers")

%md
### This is a Markdown cell, just like in Jupyter
```

This lets a single notebook load data in PySpark, analyze it with a SQL query, and
summarize the result in a Markdown cell -- all without switching between files.

**Revision history and comments.** Databricks automatically keeps a notebook's revision
history (similar to Git commits) -- you can revert to any earlier version. You can also
leave comments on specific cells, which makes code review within a team easier (much like
comments on a GitHub Pull Request).

**`%run`.** The `%run ./other_notebook` magic command runs another notebook "in place" --
all of its variables and functions become available in the current notebook. This is a
simple way to share code (e.g. common helper functions) between notebooks without
building a separate Python package.

> ⚡ **This course in Databricks**
>
> All our notebooks from Modules 1-13 are written in plain PySpark (cells with no `%magic` prefix = default Python), so imported into Databricks they'll run unchanged -- Databricks simply treats every cell as `%python`.

<a id="sec3"></a>
## 3. Clusters and Compute

Before running any code, a notebook must be **attached** to a running **cluster**
(compute) -- that's what actually performs the computations we did on a single local
machine throughout this course.

**Compute types:**

- **All-purpose cluster** -- a cluster for interactive notebook work: you start it
  manually, work on it, and it stays "alive" (and billed) until you stop it or until
  auto-termination kicks in (see below).
- **Job cluster** -- a cluster created AUTOMATICALLY for a single scheduled job (Job,
  section 6) and destroyed right after it finishes -- cheaper, since it doesn't sit idle
  between runs.
- **SQL Warehouse** -- compute optimized for SQL queries (e.g. from dashboards or BI
  tools), with its own query performance-tuning engine.
- **Serverless** -- the simplest option (and the default in Free Edition): you don't
  configure machines, instance types, or scaling at all -- the platform manages it
  automatically behind the scenes.

**Cluster configuration** (when NOT using serverless) includes: **Databricks Runtime**
(the Spark version plus a set of preconfigured libraries -- there's a separate *ML
Runtime* with TensorFlow/PyTorch/MLlib ready to go), instance type (how much CPU/RAM per
machine), **autoscaling** (the cluster adds and removes executors on its own based on
load, exactly the cluster architecture idea we covered conceptually in Module 2),
**spot/preemptible instances** (cheaper, interruptible machines for workloads that
tolerate restarts), and **auto-termination** (the cluster shuts itself down after N
minutes of idleness, so you don't pay for a machine sitting idle).

**Photon** is Databricks' accelerated, native execution engine (written in C++, not the
JVM) -- an optional speed-up for SQL and DataFrame queries, transparent to your code (you
don't change a single line of PySpark to benefit from it).

> ⚠️ **Clusters cost money (even idle)**
>
> An all-purpose cluster is billed for as long as it's running -- REGARDLESS of whether it's actually computing anything at that moment. Auto-termination (e.g. "stop after 30 minutes idle") is one of the first settings worth checking before you leave a notebook running overnight.

> 🧱 **Cluster policies**
>
> In a company environment, administrators typically define *cluster policies* -- ready-made configuration templates (e.g. "max 4 nodes, spot instances only, auto-terminate after 20 minutes") that regular users pick from instead of configuring a cluster from scratch. This is how organizations control cost and keep configuration consistent.

<a id="sec4"></a>
## 4. `dbutils` — the notebook's Swiss Army knife

`dbutils` is a special object automatically available in EVERY Databricks notebook (just
like `spark`) -- a toolbox for tasks that aren't convenient to do with plain PySpark
alone. Four of the most important submodules:

**`dbutils.fs`** -- file system operations (DBFS -- Databricks File System -- and mounted
cloud storage):

```
dbutils.fs.ls("/mnt/data")            # list files, like `ls` in a terminal
dbutils.fs.cp("source", "destination")   # copy
dbutils.fs.mv("old_path", "new_path")
dbutils.fs.rm("path", recurse=True)
```

**`dbutils.notebook`** -- running and chaining notebooks from code (not just from the
Jobs UI):

```
# runs another notebook and WAITS for the result (unlike %run, which merges
# namespaces -- .run() treats it like a "subroutine")
result = dbutils.notebook.run("./process_data", timeout_seconds=600, arguments={"data": "2024-01-01"})

# at the END of the notebook that gets called this way:
dbutils.notebook.exit("success: processed 3000 rows")
```

This is the foundation for building **pipelines** out of several smaller notebooks,
instead of one giant file.

**`dbutils.secrets`** -- secure access to secrets (passwords, API keys) without typing
them directly into notebook code:

```
password = dbutils.secrets.get(scope="my-team", key="db_password")
```

Secrets are entered separately (via the CLI or API) into a *secret scope* -- their values
are never visible in the notebook itself or in its revision history, which is exactly the
security concern we covered around `.env` files and environment variables in the Python
course.

**`dbutils.widgets`** -- covered separately in section 5, since it's one of the most
commonly used submodules in practice.

> ⚡ **`dbutils.fs` vs. `%fs`**
>
> The same file operations can also be called as a magic command in a cell: `%fs ls /mnt/data` instead of `dbutils.fs.ls("/mnt/data")`. The `%fs` output displays right away as a readable table -- convenient for quick exploration, while `dbutils.fs.ls()` is more convenient when you want to process the result further in code.

<a id="sec5"></a>
## 5. Widgets — parameterizing notebooks

**Widgets** (`dbutils.widgets`) turn a notebook from a single fixed script into a
**parameterizable template** -- the same notebook can be run for different values (e.g.
different dates, environments, customer segments) without changing a single line of code.

```
# at the top of the notebook -- we define a widget (once; subsequent runs don't overwrite it)
dbutils.widgets.text("report_date", "2024-01-01", "Report date")
dbutils.widgets.dropdown("environment", "dev", ["dev", "test", "production"], "Environment")
dbutils.widgets.multiselect("segments", "Premium", ["New", "Standard", "Premium"], "Segments")

# anywhere below -- we read the value
report_date = dbutils.widgets.get("report_date")
environment = dbutils.widgets.get("environment")

orders_for_day = orders.filter(orders.order_date == report_date)
```

In the Databricks UI these widgets appear as text fields / dropdown lists at the top of
the notebook -- someone who doesn't know Spark can run the notebook with different
parameters just by changing a field's value, instead of editing code.

Widget types: `text` (any text), `dropdown` (a single value from a list), `combobox` (a
list plus the ability to type a custom value), `multiselect` (several values at once).

> 🧱 **Widgets + Jobs = parameterized production tasks**
>
> The real power of widgets shows up combined with Jobs (section 6): when defining a job, you supply CONCRETE parameter values (e.g. `report_date = {{job.start_time}}` -- a special variable holding the job's run date), so the same notebook, run every day at 6:00 AM, automatically processes the right day's data -- with no manual editing.

<a id="sec6"></a>
## 6. Jobs and Workflows — scheduling tasks

**Jobs** (in the newer UI: **Workflows**) is Databricks' mechanism for running notebooks
and scripts AUTOMATICALLY, without manually clicking "Run" -- the foundation of
production data pipelines.

**Multi-task jobs.** A single job can consist of multiple **tasks** arranged in a
dependency graph (DAG) -- e.g. Task A ("load and clean data") must finish successfully
before Task B ("compute aggregates") starts, and Task B must finish before Task C ("train
an MLlib model") and Task D ("write a SQL report") start IN PARALLEL. This is exactly the
same idea as orchestrating multiple notebooks via `dbutils.notebook.run()` (section 4),
but managed visually, with run history, retries, and alerts built into the platform.

**Triggers** -- what starts a job:

- **Schedule** -- a cron-style expression, e.g. "every day at 6:00 AM".
- **File arrival** -- the job starts automatically when a new file appears in a monitored
  location (typical for event-driven pipelines).
- **Continuous** -- the job runs non-stop, restarting immediately after finishing
  (typical for stream processing).
- **Manual / on demand** -- triggered by a person, or by an API/CLI call.

**Job cluster** (mentioned in section 3) is the default compute type for jobs: created
when the job starts, destroyed once it finishes -- cheaper than keeping an all-purpose
cluster running all the time just waiting for the daily run.

**Retries and alerts.** Every task can be configured to automatically retry on failure
(e.g. 3 times, with a 5-minute gap) and to send a notification (email, Slack) on success,
failure, or timeout.

> 💡 **Databricks Asset Bundles (DAB)**
>
> For teams that want to keep job, cluster, and notebook configuration as code (Infrastructure as Code) and deploy it via CI/CD -- similar to Terraform for cloud infrastructure -- Databricks offers **Asset Bundles**: YAML files describing an entire project, deployed with the `databricks bundle deploy` command. This is beyond the scope of this course, but worth knowing it exists as a team matures from individual notebooks to a full deployment process.

<a id="sec7"></a>
## 7. A hands-on walkthrough of Databricks Free Edition

Time to turn the theory above into practice in your own **Databricks Free Edition**
account (created in Module 1). Work through the following steps yourself in the browser:

1. **Import a notebook.** In Workspace, right-click your folder → *Import* → upload any
   `.ipynb` file from this course (e.g. `Spark_Module3...ipynb`). Databricks will open it
   exactly as it looks here.
2. **Create and attach compute.** In the *Compute* section, create a new cluster (or use
   the default serverless option) and attach it to the imported notebook (the *Connect*
   button in the notebook's top-right corner).
3. **Run the notebook.** *Run All* executes every cell in order -- you should see
   identical results to what you saw working locally throughout this course.
4. **Add a widget.** In a new, empty cell at the top of the notebook, type
   `dbutils.widgets.text("segment_filter", "Premium", "Segment")` and run the cell -- a
   text field appears at the top of the notebook. Change its value, then in the next cell
   read it: `dbutils.widgets.get("segment_filter")`.
5. **Create a simple Job.** In the *Workflows* section, create a new job with a single
   task pointing to your notebook, set a schedule (e.g. "once daily at 8:00 AM"), and
   save -- you don't need to wait for a run; just seeing the configuration screen is
   enough to see how the task DAG and trigger choice work.

> ⚡ **Don't have an account yet?**
>
> Go back to Module 1, section 6 -- there we described step by step how to create a free Databricks Free Edition account (all you need is an email address, no cloud account of your own).

<a id="sec8"></a>
## 8. Summary and exercises

In this module we covered the Databricks platform (not new Spark syntax):
- the anatomy of the workspace: Workspace, Repos, Catalog, Compute, Workflows,
- multi-language notebooks (`%python`/`%sql`/`%scala`/`%md`), revision history, and `%run`,
- compute types (all-purpose, job cluster, SQL Warehouse, serverless) and their
  configuration (Runtime, autoscaling, spot instances, auto-termination, Photon),
- `dbutils` -- `fs`, `notebook`, `secrets`, and (in depth) `widgets`,
- parameterizing notebooks with widgets,
- Jobs/Workflows: multi-task jobs, triggers, retries, and alerts,
- your first practical steps in your own Databricks Free Edition account.

Do the exercises below in your own Databricks Free Edition account -- they can't be
checked locally, since they require a real `dbutils` and the Jobs UI.

### 📝 Exercise 1: Import and run

In Databricks Free Edition, import the Module 5 notebook (Spark SQL) and run it entirely (*Run All*) on attached compute. Check whether the last cell's result matches what you saw running it locally.

<details>
<summary>Show solution</summary>

```text
1. Workspace -> your folder -> Import -> select the Module 5 .ipynb file.
2. Compute -> create/select a cluster (or use the default serverless option).
3. In the opened notebook: click "Connect" in the top-right corner -> pick the compute.
4. Run All. The last cell's result (e.g. the printed columns from
   spark.catalog.listColumns) should be identical to the local run in this course.
```

</details>

### 📝 Exercise 2: Add two widgets

In any imported notebook, add a `dropdown` widget with a list of cities (`Warsaw`, `Krakow`, `Gdansk`, `Wroclaw`, `Poznan`) and a `text` widget with a default segment value (`Premium`). Read both values and use them to filter the `customers` DataFrame.

<details>
<summary>Show solution</summary>

```python
dbutils.widgets.dropdown("city", "Warsaw", ["Warsaw", "Krakow", "Gdansk", "Wroclaw", "Poznan"], "City")
dbutils.widgets.text("segment", "Premium", "Segment")

city = dbutils.widgets.get("city")
segment = dbutils.widgets.get("segment")

customers.filter((customers.city == city) & (customers.segment == segment)).show()
```

</details>

### 📝 Exercise 3: Schedule a job

In the Workflows section, create a new Job with a single task pointing to the notebook from exercise 2. Set the schedule to "every day at 6:00 AM" and turn on an email notification for job failure.

<details>
<summary>Show solution</summary>

```text
1. Workflows -> Create Job.
2. Task: choose "Notebook" as the type, point it to the notebook from Exercise 2, pick a
   compute (job cluster or an existing one).
3. Schedule -> Add trigger -> Scheduled -> cron: "0 6 * * *" (every day at 6:00 AM).
4. Notifications -> add your email for the "On failure" event.
5. Save. (Actually running the job is not required to complete the exercise.)
```

</details>

> 🔥 **Challenge: a mini pipeline of two notebooks**
>
> Create a SECOND notebook that accepts a `min_value` widget (type `text`, default value `"100"`) and prints the number of orders whose value exceeds it (you'll need the `value` column from Module 12 -- recompute it by joining orders with order items). In the FIRST notebook, call the second one via `dbutils.notebook.run("./value_threshold", 300, {"min_value": "500"})` and print the returned result.

<details>
<summary>Show solution</summary>

```python
# Notebook 2 ("value_threshold"):
dbutils.widgets.text("min_value", "100")
threshold = float(dbutils.widgets.get("min_value"))

from pyspark.sql import functions as F
orders_with_value = (
    order_items.groupBy("order_id")
    .agg(F.sum(F.col("quantity") * F.col("unit_price") * (1 - F.col("discount"))).alias("value"))
)
count = orders_with_value.filter(F.col("value") > threshold).count()
dbutils.notebook.exit(f"Orders above {threshold}: {count}")

# Notebook 1 (the caller):
result = dbutils.notebook.run("./value_threshold", 300, {"min_value": "500"})
print(result)
```

dbutils.notebook.run() returns exactly what was passed to .exit() in the second notebook -- the simplest way to pass results between notebooks without shared global variables.

</details>

## What's next?

In **Module 15**, the course's final module, we'll bring EVERYTHING together -- from
loading raw data, through cleaning and transforming it with the DataFrame API, Spark SQL
analysis, and training an MLlib model, to writing the results out (partitioned Parquet) --
into one complete, runnable end-to-end data pipeline.